# FEA Sequence-Order Test Prediction Collection

This notebook exports aligned test-set predictions for the three final FEA sequence-order ablation models:

- **Ordered LSTM** — chronological 30-observation FEA sequences.
- **Shuffled LSTM** — the same 30 observations in a deterministic random order using `SEED + 2` for the test set.
- **Mean aggregation** — temporal mean of the same 30 FEA observations.

The output contains one row per reenactment (378 rows) and is intended as the common input for the Ordered-vs.-Shuffled and Ordered-vs.-Mean significance-test notebooks.

## 1. Setup

### 1.0 Add log filter

Add the log filter below or tensorflow will print thousands of lines of uninformative log messages.  


In [1]:
import re
import ipykernel.iostream

TF_LOG_FILTER_PATTERNS = [
    r'ptx\d+.*is not a recognized feature for this target',
    r'is not a recognized feature for this target \(ignoring feature\)',
    r'\(ignoring feature\)',
    r'successful NUMA node read from SysFS had negative value \(-1\)',
    r'gpu_timer\.cc:114\] Skipping the delay kernel, measurement accuracy will be reduced',
]

KERAS_PROGRESS_PATTERNS = [
    r'ms/step',
    r's/step',
    r'ETA:',
    r'\d+/\d+ \[',   # 12/64 [===>...]
]

_original_write = ipykernel.iostream.OutStream.write

def _filtered_write(self, msg, *args, **kwargs):
    text = str(msg)

    if any(re.search(p, text) for p in KERAS_PROGRESS_PATTERNS):
        _original_write(self, text, *args, **kwargs)
        return

    buf = getattr(self, '_tf_log_filter_buf', '')
    buf += text

    if '\n' not in buf:
        setattr(self, '_tf_log_filter_buf', buf)
        return

    lines = buf.splitlines(keepends=True)
    if not buf.endswith('\n'):
        incomplete = lines.pop()
    else:
        incomplete = ''

    for line in lines:
        if any(re.search(p, line) for p in TF_LOG_FILTER_PATTERNS):
            continue
        _original_write(self, line, *args, **kwargs)

    setattr(self, '_tf_log_filter_buf', incomplete)

ipykernel.iostream.OutStream.write = _filtered_write

print('Notebook log filter installed (targeted, keeps Keras steps).')


Notebook log filter installed (targeted, keeps Keras steps).


### 1.1 Imports and paths

The dataset and ordered-model paths follow the existing `emohevrdb-dfer` Docker/repository conventions. Place the final optimized shuffled and mean models under the filenames below, or adjust only the corresponding path constants.

In [13]:
from pathlib import Path
import gc

import keras
import numpy as np
import pandas as pd
import tensorflow as tf


DATASET_ROOT = Path('/workspace/datasets')
MODEL_ROOT = Path('../models')

DFEA_TEST_PATH = DATASET_ROOT / 'emoji-hero-vr-db-dfea-as-csv' / 'test_set.csv'

ORDERED_MODEL_PATH = MODEL_ROOT / 'fea_sequence_model.keras'
SHUFFLED_MODEL_PATH = MODEL_ROOT / 'fea_sequence_shuffled_model.keras'
MEAN_MODEL_PATH = MODEL_ROOT / 'fea_sequence_mean_model.keras'

SEED = 31
SEQUENCE_LENGTH = 30
N_FEATURES = 63
BATCH_SIZE = 32
TEST_SEQUENCE_SHUFFLE_SEED = SEED + 2

print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

for path in [DFEA_TEST_PATH, ORDERED_MODEL_PATH, SHUFFLED_MODEL_PATH, MEAN_MODEL_PATH]:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")

TensorFlow: 2.17.0
Keras: 3.12.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
/workspace/datasets/emoji-hero-vr-db-dfea-as-csv/test_set.csv: OK
models/fea_sequence_model.keras: OK
models/fea_sequence_shuffled_model.keras: OK
models/fea_sequence_mean_model.keras: OK


### 1.1 Class mapping

In [14]:
ID_TO_EMOTION = {
    0: 'Anger',
    1: 'Disgust',
    2: 'Fear',
    3: 'Happiness',
    4: 'Neutral',
    5: 'Sadness',
    6: 'Surprise'
}

EMOTIONS = list(ID_TO_EMOTION.values())

## 2. Load and prepare the FEA test set

The ordered sequences are constructed exactly as in the original FEA notebook: group by `sequence_id`, sort each group by `timestamp`, and retain all 63 FEA columns.

In [15]:
test_df = pd.read_csv(DFEA_TEST_PATH)

print('test_df.shape:', test_df.shape)
print('Columns:', test_df.columns.tolist())

FEA_COLUMNS = test_df.columns[2:-1].tolist()

assert len(FEA_COLUMNS) == N_FEATURES
assert test_df['sequence_id'].nunique() == 378
assert (test_df.groupby('sequence_id').size() == SEQUENCE_LENGTH).all()
assert test_df[FEA_COLUMNS].notna().all().all()

test_df.shape: (11340, 66)
Columns: ['sequence_id', 'timestamp', 'BrowLowererL', 'BrowLowererR', 'CheekPuffL', 'CheekPuffR', 'CheekRaiserL', 'CheekRaiserR', 'CheekSuckL', 'CheekSuckR', 'ChinRaiserB', 'ChinRaiserT', 'DimplerL', 'DimplerR', 'EyesClosedL', 'EyesClosedR', 'EyesLookDownL', 'EyesLookDownR', 'EyesLookLeftL', 'EyesLookLeftR', 'EyesLookRightL', 'EyesLookRightR', 'EyesLookUpL', 'EyesLookUpR', 'InnerBrowRaiserL', 'InnerBrowRaiserR', 'JawDrop', 'JawSidewaysLeft', 'JawSidewaysRight', 'JawThrust', 'LidTightenerL', 'LidTightenerR', 'LipCornerDepressorL', 'LipCornerDepressorR', 'LipCornerPullerL', 'LipCornerPullerR', 'LipFunnelerLB', 'LipFunnelerLT', 'LipFunnelerRB', 'LipFunnelerRT', 'LipPressorL', 'LipPressorR', 'LipPuckerL', 'LipPuckerR', 'LipStretcherL', 'LipStretcherR', 'LipSuckLB', 'LipSuckLT', 'LipSuckRB', 'LipSuckRT', 'LipTightenerL', 'LipTightenerR', 'LipsToward', 'LowerLipDepressorL', 'LowerLipDepressorR', 'MouthLeft', 'MouthRight', 'NoseWrinklerL', 'NoseWrinklerR', 'OuterBro

In [16]:
def prepare_data(df):
    grouped_by_sequence = df.groupby('sequence_id')

    sequences = []
    labels = []
    sequence_ids = []

    for sequence_id, group in grouped_by_sequence:
        sorted_group = group.sort_values(by='timestamp')
        sequences.append(sorted_group.iloc[:, 2:-1].values)
        labels.append(sorted_group.iloc[0, -1])
        sequence_ids.append(sequence_id)

    X = np.array(sequences)
    y = np.array(labels)
    return X, y, sequence_ids


X_test_ordered, y_test, reenactment_ids = prepare_data(test_df)

assert X_test_ordered.shape == (378, SEQUENCE_LENGTH, N_FEATURES)
assert y_test.shape == (378,)
assert len(reenactment_ids) == 378

print('X_test_ordered.shape:', X_test_ordered.shape)
print('y_test.shape:', y_test.shape)

X_test_ordered.shape: (378, 30, 63)
y_test.shape: (378,)


### 2.1 Reenactment metadata

In [17]:
def parse_reenactment_id(reenactment_id: str) -> dict:
    parts = str(reenactment_id).split('-')

    if len(parts) != 6:
        raise ValueError(f'Unexpected reenactment ID: {reenactment_id}')

    timestamp, set_id, participant_id, level_id, emoji_id, emotion_id = parts

    return {
        'reenactment_id': str(reenactment_id),
        'timestamp': int(timestamp),
        'set_id': int(set_id),
        'participant_id': int(participant_id),
        'level_id': int(level_id),
        'emoji_id': int(emoji_id),
        'true_label_id_from_id': int(emotion_id)
    }


prediction_df = pd.DataFrame([parse_reenactment_id(reenactment_id) for reenactment_id in reenactment_ids])
prediction_df['true_label_id'] = y_test.astype(int)
prediction_df['true_label'] = prediction_df['true_label_id'].map(ID_TO_EMOTION)

assert len(prediction_df) == 378
assert prediction_df['reenactment_id'].is_unique
assert prediction_df['participant_id'].nunique() == 8
assert (prediction_df['true_label_id'] == prediction_df['true_label_id_from_id']).all()
assert (prediction_df.groupby('true_label_id').size() == 54).all()

prediction_df = prediction_df.drop(columns='true_label_id_from_id')

display(prediction_df.head())

,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,true_label_id,true_label
0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Anger
1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,5,Sadness
2,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,3,Happiness
3,1700479004312-2-1-1-3-0,1700479004312,2,1,1,3,0,Anger
4,1700479005401-2-1-1-4-0,1700479005401,2,1,1,4,0,Anger


## 3. Create and verify the deterministic shuffled test sequences

This uses the same shuffling implementation as the FEA ablation notebook. The order of `X_test_ordered` is kept unchanged before shuffling because the deterministic RNG assigns successive permutations to successive sequences.

In [18]:
def shuffle_sequences(X, seed):
    rng = np.random.default_rng(seed)

    X_shuffled = X.copy()

    for i in range(len(X_shuffled)):
        permutation = rng.permutation(X_shuffled.shape[1])
        X_shuffled[i] = X_shuffled[i, permutation]

    return X_shuffled


X_test_shuffled = shuffle_sequences(X_test_ordered, TEST_SEQUENCE_SHUFFLE_SEED)

In [19]:
def verify_shuffling(X_ordered, X_shuffled):

    # The shuffled dataset must preserve the exact overall array dimensions.
    assert X_ordered.shape == X_shuffled.shape

    for i in range(len(X_ordered)):
        ordered_sequence = X_ordered[i]
        shuffled_sequence = X_shuffled[i]

        # Each sequence must still contain exactly 30 observations.
        assert len(ordered_sequence) == SEQUENCE_LENGTH
        assert len(shuffled_sequence) == SEQUENCE_LENGTH

        unique_ordered, counts_ordered = np.unique(
            ordered_sequence, axis=0, return_counts=True
        )
        unique_shuffled, counts_shuffled = np.unique(
            shuffled_sequence, axis=0, return_counts=True
        )

        # The shuffled sequence must contain exactly the same observations
        # with the same multiplicities as the ordered sequence.
        assert np.array_equal(unique_ordered, unique_shuffled) and np.array_equal(counts_ordered, counts_shuffled)

        # The observation order must actually differ after shuffling.
        assert not np.array_equal(ordered_sequence, shuffled_sequence)

    print(f'Verified {len(X_ordered)} shuffled sequences.')


verify_shuffling(X_test_ordered, X_test_shuffled)

# The same seed must reproduce exactly the same shuffled test set.
assert np.array_equal(
    X_test_shuffled,
    shuffle_sequences(X_test_ordered, TEST_SEQUENCE_SHUFFLE_SEED)
)

print('Verified deterministic test-set shuffling.')

Verified 378 shuffled sequences.
Verified deterministic test-set shuffling.


## 4. Prepare model inputs and prediction storage

The ordered and shuffled LSTMs receive `(30, 63)` sequences. The mean model receives one 63-dimensional temporal mean per reenactment.

In [20]:
X_test_mean = np.mean(X_test_ordered, axis=1)

assert X_test_mean.shape == (378, N_FEATURES)

print('Ordered input:', X_test_ordered.shape)
print('Shuffled input:', X_test_shuffled.shape)
print('Mean input:', X_test_mean.shape)

Ordered input: (378, 30, 63)
Shuffled input: (378, 30, 63)
Mean input: (378, 63)


In [21]:
def create_fea_dataset(X, batch_size=BATCH_SIZE):
    return tf.data.Dataset.from_tensor_slices(X.astype(np.float32)).batch(batch_size).prefetch(1)


def add_predictions(df: pd.DataFrame, prefix: str, probabilities: np.ndarray) -> np.ndarray:
    probabilities = np.asarray(probabilities)

    assert probabilities.shape == (len(df), 7)
    assert np.all(probabilities >= 0)
    assert np.all(probabilities <= 1)
    assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4)

    predictions = np.argmax(probabilities, axis=1)

    df[f'{prefix}_pred_id'] = predictions
    df[f'{prefix}_pred'] = [ID_TO_EMOTION[p] for p in predictions]

    for i, emotion in enumerate(EMOTIONS):
        df[f'{prefix}_prob_{emotion.lower()}'] = probabilities[:, i]

    return predictions

## 5. Ordered LSTM

Expected test result: **296 / 378 = 78.31%**.

In [22]:
ordered_model = keras.models.load_model(ORDERED_MODEL_PATH, compile=False)

print('Input:', ordered_model.input_shape)
print('Output:', ordered_model.output_shape)

assert ordered_model.input_shape[1:] == (SEQUENCE_LENGTH, N_FEATURES)

ordered_probabilities = ordered_model.predict(create_fea_dataset(X_test_ordered), verbose=1)
ordered_predictions = add_predictions(prediction_df, 'ordered', ordered_probabilities)

ordered_correct = np.sum(ordered_predictions == y_test)
ordered_accuracy = ordered_correct / len(y_test)

print(f'Ordered: {ordered_correct}/378 = {ordered_accuracy:.6f}')
assert ordered_correct == 296

Input: (None, 30, 63)
Output: (None, 7)
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step  
Ordered: 296/378 = 0.783069


In [23]:
del ordered_model, ordered_probabilities
keras.backend.clear_session()
gc.collect()

0

## 6. Shuffled LSTM

The optimized shuffled LSTM is evaluated on the deterministic test-set permutation generated with `SEED + 2 = 33`.

Expected test result: **265 / 378 = 70.11%**.

In [24]:
shuffled_model = keras.models.load_model(SHUFFLED_MODEL_PATH, compile=False)

print('Input:', shuffled_model.input_shape)
print('Output:', shuffled_model.output_shape)

assert shuffled_model.input_shape[1:] == (SEQUENCE_LENGTH, N_FEATURES)

shuffled_probabilities = shuffled_model.predict(create_fea_dataset(X_test_shuffled), verbose=1)
shuffled_predictions = add_predictions(prediction_df, 'shuffled', shuffled_probabilities)

shuffled_correct = np.sum(shuffled_predictions == y_test)
shuffled_accuracy = shuffled_correct / len(y_test)

print(f'Shuffled: {shuffled_correct}/378 = {shuffled_accuracy:.6f}')
assert shuffled_correct == 265

Input: (None, 30, 63)
Output: (None, 7)
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
Shuffled: 265/378 = 0.701058


In [25]:
del shuffled_model, shuffled_probabilities
keras.backend.clear_session()
gc.collect()

0

## 7. Mean aggregation

The optimized mean model is evaluated on the temporal mean of the original chronological 30-observation sequence.

Expected test result: **261 / 378 = 69.05%**.

In [26]:
mean_model = keras.models.load_model(MEAN_MODEL_PATH, compile=False)

print('Input:', mean_model.input_shape)
print('Output:', mean_model.output_shape)

assert mean_model.input_shape[1:] == (N_FEATURES,)

mean_probabilities = mean_model.predict(create_fea_dataset(X_test_mean), verbose=1)
mean_predictions = add_predictions(prediction_df, 'mean', mean_probabilities)

mean_correct = np.sum(mean_predictions == y_test)
mean_accuracy = mean_correct / len(y_test)

print(f'Mean: {mean_correct}/378 = {mean_accuracy:.6f}')
assert mean_correct == 261

Input: (None, 63)
Output: (None, 7)
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
Mean: 261/378 = 0.690476


I0000 00:00:1789752276.953949     948 service.cc:146] XLA service 0x77f148006ed0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789752276.953992     948 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce RTX 5090, Compute Capability 12.0
I0000 00:00:1789752277.014823     948 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [27]:
del mean_model, mean_probabilities
keras.backend.clear_session()
gc.collect()

0

## 8. Final validation and export

In [28]:
assert len(prediction_df) == 378
assert prediction_df['reenactment_id'].is_unique
assert prediction_df['participant_id'].nunique() == 8
assert (prediction_df.groupby('true_label_id').size() == 54).all()

assert prediction_df[['ordered_pred_id', 'shuffled_pred_id', 'mean_pred_id']].notna().all().all()

for prefix in ['ordered', 'shuffled', 'mean']:
    probability_columns = [f'{prefix}_prob_{emotion.lower()}' for emotion in EMOTIONS]
    assert prediction_df[probability_columns].notna().all().all()
    assert np.allclose(prediction_df[probability_columns].sum(axis=1), 1.0, atol=1e-4)

print('Final prediction table passed all integrity checks.')

Final prediction table passed all integrity checks.


In [29]:
for model in ['ordered', 'shuffled', 'mean']:
    correct = (prediction_df[f'{model}_pred_id'] == prediction_df['true_label_id']).sum()
    print(f'{model:<10}: {correct:>3}/378 = {correct / 378:.4%}')

ordered   : 296/378 = 78.3069%
shuffled  : 265/378 = 70.1058%
mean      : 261/378 = 69.0476%


### 8.1 Export `fea_test_predictions.csv`

In [30]:
output_columns = [
    'reenactment_id',
    'timestamp',
    'set_id',
    'participant_id',
    'level_id',
    'emoji_id',
    'true_label_id',
    'true_label'
]

prediction_columns = [
    c for c in prediction_df.columns
    if c.startswith('ordered_') or c.startswith('shuffled_') or c.startswith('mean_')
]

output_df = prediction_df[output_columns + prediction_columns].copy()
output_df.to_csv('fea_test_predictions.csv', index=False)

print(f'Exported {len(output_df)} rows to fea_test_predictions.csv')
display(output_df.head())

Exported 378 rows to fea_test_predictions.csv


,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,true_label_id,true_label,ordered_pred_id,ordered_pred,...,shuffled_prob_surprise,mean_pred_id,mean_pred,mean_prob_anger,mean_prob_disgust,mean_prob_fear,mean_prob_happiness,mean_prob_neutral,mean_prob_sadness,mean_prob_surprise
0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Anger,0,Anger,...,0.003162,0,Anger,0.838640,0.034060,0.010857,6.689494e-04,0.001781,0.086494,0.027500
1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,5,Sadness,5,Sadness,...,0.000900,5,Sadness,0.000826,0.000346,0.000051,5.587236e-07,0.000006,0.998732,0.000038
2,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,3,Happiness,3,Happiness,...,0.040769,3,Happiness,0.000762,0.017813,0.249371,5.898122e-01,0.000365,0.009472,0.132405
3,1700479004312-2-1-1-3-0,1700479004312,2,1,1,3,0,Anger,4,Neutral,...,0.010360,4,Neutral,0.096489,0.024780,0.020434,1.909493e-02,0.783427,0.018887,0.036888
4,1700479005401-2-1-1-4-0,1700479005401,2,1,1,4,0,Anger,0,Anger,...,0.003727,0,Anger,0.815663,0.054917,0.007815,1.686807e-03,0.008238,0.085832,0.025849
